# Kaggle MCP Example

This notebook demonstrates how to use the NeMo Agent Toolkit SDK to integrate with Kaggle's MCP server, enabling interaction with Kaggle's datasets, notebooks, models, and competitions.

## Key Features

1. **Remote MCP Server** - Connect to Kaggle's public MCP server
2. **Bearer Token Authentication** - Use API key authentication with Bearer scheme
3. **Tool Overrides** - Improve tool descriptions for better LLM responses

## Prerequisites

1. A Kaggle account and API token from [Kaggle Account Settings](https://www.kaggle.com/settings/account)

2. Set environment variables:
   - `NVIDIA_API_KEY` - NVIDIA API key
   - `KAGGLE_BEARER_TOKEN` - Your Kaggle API token


In [ ]:
import os
import sys

# Add src to path for development
module_path = os.path.abspath('../../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)


## Creating the Workflow

We'll create a workflow that connects to Kaggle's MCP server with Bearer token authentication and uses tool overrides to improve the LLM's understanding of the API.


In [ ]:
from pathlib import Path

from pydantic import HttpUrl
from pydantic import SecretStr

from nat.agent.react_agent.register import NatReActAgent
from nat.authentication.api_key.api_key_auth_provider_config import APIKeyAuth
from nat.data_models.authentication import HeaderAuthScheme
from nat.llm.nim_llm import NimLLM
from nat.plugins.mcp.client_config import MCPClient
from nat.plugins.mcp.client_config import MCPServerConfig
from nat.plugins.mcp.client_config import MCPToolOverrideConfig
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create the LLM
llm = NimLLM(
    model_name="meta/llama-3.1-70b-instruct",
    temperature=0.0,
    name="nim_llm",
)

# Create authentication provider for Kaggle
# Note: KAGGLE_BEARER_TOKEN environment variable must be set
kaggle_auth = APIKeyAuth(
    raw_key=SecretStr(os.environ.get("KAGGLE_BEARER_TOKEN", "")),
    auth_scheme=HeaderAuthScheme.BEARER,
    name="kaggle",
)

# Create MCP client for Kaggle
# Tool overrides help the LLM understand the correct parameter names
kaggle_mcp_tools = MCPClient(
    server=MCPServerConfig(
        transport="streamable-http",
        url=HttpUrl("https://www.kaggle.com/mcp"),
        auth_provider="kaggle",
    ),
    tool_overrides={
        "search_datasets": MCPToolOverrideConfig(
            description=(
                "Search for datasets on Kaggle. Use the 'search' parameter to search "
                "by keywords. Returns a list of datasets with metadata including title, "
                "owner, download count, and URL. Example: {\"request\": {\"search\": \"titanic\"}}"
            ),
        ),
    },
    name="kaggle_mcp_tools",
)

# Create the ReAct agent with Kaggle MCP tools
agent = NatReActAgent(
    tools=[kaggle_mcp_tools],
    llm=llm,
    verbose=True,
    parse_agent_response_max_retries=3,
)

# Wrap in NatWorkflow
nat_workflow = NatWorkflow(
    entrypoint=agent,
)

# Add the authentication provider
nat_workflow.add_auth_provider(kaggle_auth)

print("Workflow created successfully!")


## Saving the Configuration

Save the workflow configuration to a YAML file.


In [ ]:
config_dir = Path(os.getcwd()) / "config"
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "workflow_config.yaml"
nat_workflow.save_to_config_file(config_path)

print(f"Configuration saved to: {config_path}")
print("\n" + "="*50 + "\n")

with open(config_path) as f:
    print(f.read())


## Testing the Workflow

Test the workflow with a Kaggle query.

**Note**: Make sure `KAGGLE_BEARER_TOKEN` is set to your Kaggle API token.


In [ ]:
# Test the workflow (requires KAGGLE_BEARER_TOKEN)
# Uncomment to run:
# result = await nat_workflow.prompt(
#     "Find the most popular datasets about natural language processing"
# )
# print(result)


In [ ]:
# Additional test queries
# Uncomment to run:

# Dataset information
# result = await nat_workflow.prompt("What is the titanic dataset about?")
# print(result)

# Competition search
# result = await nat_workflow.prompt("What competitions are currently active?")
# print(result)

# Model search
# result = await nat_workflow.prompt("Find machine learning models for image classification")
# print(result)


## Summary

This notebook demonstrated:

1. **MCPClient SDK Class** - Connecting to Kaggle's public MCP server
2. **ApiKeyAuthProviderConfig** - Bearer token authentication for MCP servers
3. **Tool Overrides** - Improving LLM accuracy by providing detailed parameter guidance
4. **Authentication Integration** - Passing auth providers to NatWorkflow

### Authentication Configuration

The workflow uses `ApiKeyAuthProviderConfig` with Bearer scheme:

| Field | Description |
|---|---|
| `raw_key` | The Kaggle API token (from environment variable) |
| `auth_scheme` | "Bearer" - the authentication scheme |
| `name` | "kaggle" - referenced by `auth_provider` in MCP server config |

### CLI Commands

You can also use CLI commands to explore Kaggle MCP tools:

```bash
# List available tools
nat mcp client tool list --url https://www.kaggle.com/mcp

# Get tool schema
nat mcp client tool list --url https://www.kaggle.com/mcp --tool search_datasets

# Call a tool with authentication
nat mcp client tool call search_datasets \
  --url https://www.kaggle.com/mcp \
  --bearer-token-env KAGGLE_BEARER_TOKEN \
  --json-args '{"request": {"search": "titanic"}}'
```

### Related Examples

- [Simple Calculator MCP](../simple_calculator_mcp/) - MCP client with multiple transports
- [Simple Auth MCP](../simple_auth_mcp/) - OAuth2 authentication for enterprise MCP servers
